<a href="https://colab.research.google.com/github/Yike-Ding/structural_bioinformatics/blob/main/01_obtaining_histone_h3_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# Histone H3 Structural Bioinformatics Project
# Step 1: Retrieve and inspect experimentally determined structures

import sys

print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [25]:
import pandas
import numpy
import matplotlib

print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("Matplotlib:", matplotlib.__version__)

NumPy: 2.1.3
pandas: 2.2.3
Matplotlib: 3.10.0


In [26]:
%pip install rdkit
import rdkit

print("rdkit:", rdkit.__version__)

rdkit: 2026.03.6


In [27]:
import requests
from pathlib import Path

In [28]:
def download_pdb(pdb_id):
  """download a protein structure from a given url."""

  url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

  response = requests.get(url)
  response.raise_for_status()

  with open(f"{pdb_id}.pdb", "wb") as f:   # Open 3AFA.pdb and temporarily call the opened file f.
      f.write(response.content)

  print(f"Downloaded {pdb_id}.pdb")

In this work, histone 3AFA is selected and obtained from RCSB protein data bank for the strucural analysis and comparison.

In [29]:
download_pdb("3AFA")

Downloaded 3AFA.pdb


In [30]:
# verify the "3AFA.pdb" file is actually downloaded

from pathlib import Path
p = Path("3AFA.pdb")
print(p.exists(), p.stat().st_size, "bytes")

True 1062153 bytes


Visualising the nucleosome structure 3AFA:

In [31]:
%pip install py3Dmol
import py3Dmol

with open("3AFA.pdb") as f:
  pdb_data = f. read()

  # () calls the "read" method

view = py3Dmol.view(width = 500, height = 400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [52]:
%pip install biopython
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import PPBuilder

def load_structure(pdb_id, pdb_file):
  parser = PDBParser(QUIET = True)
  structure = parser.get_structure(pdb_id, pdb_file)
  return structure


In [53]:
def get_sequence(structure):

  ppb = PPBuilder()

  for model in structure:
    for chain in model:
      residues = list(chain.get_residues())  # in the context of proteins, residues = amino acid
      peptides = ppb.build_peptides(chain)
      for pep in peptides:
        seq = pep.get_sequence()
        print(f"Chain {chain.id}: {len(residues)} residues, sequence: {seq[:20]}...")


In [54]:
structure = load_structure("3AFA","3AFA.pdb")
get_sequence(structure)

Chain A: 117 residues, sequence: PHRYRPGTVALREIRRYQKS...
Chain B: 93 residues, sequence: NIQGITKPAIRRLARRGGVK...
Chain C: 128 residues, sequence: RAKAKTRSSRAGLQFPVGRV...
Chain D: 114 residues, sequence: KRSRKESYSIYVYKVLKQVH...
Chain E: 133 residues, sequence: KPHRYRPGTVALREIRRYQK...
Chain F: 110 residues, sequence: RKVLRDNIQGITKPAIRRLA...
Chain G: 120 residues, sequence: KTRSSRAGLQFPVGRVHRLL...
Chain H: 105 residues, sequence: RKESYSIYVYKVLKQVHPDT...


We then look for the amino acid sequence of human histone H3.1 from Uniprot and compare it with the 10 polypeptide chains (Chain A to H).

A function is defined to search for what the accession number to human histone H3.1 is.

In [55]:
def search_uniprot(query, organism="human"):
    """
    Search UniProt by protein name to find candidate accession numbers.
    """
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"{query} AND organism_name:{organism}",
        "format": "json",
        "fields": "accession,protein_name,gene_names",
        "size": 5
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json()["results"]

    for r in results:
        acc = r["primaryAccession"]
        name = r["proteinDescription"]["recommendedName"]["fullName"]["value"]
        print(f"{acc}: {name}")

search_uniprot("Histone H3.1")

P68431: Histone H3.1
Q9H9B1: Histone-lysine N-methyltransferase EHMT1
O43463: Histone-lysine N-methyltransferase SUV39H1
Q9UGL1: Lysine-specific demethylase 5B
Q8TEK3: Histone-lysine N-methyltransferase, H3 lysine-79 specific


Given the accession number to human histone H3.1 is P68431:

In [56]:
def fetch_uniprot_sequence(accession_number):

  '''fetch the amino acid of a protein from UniProt given its accession number.'''

  url = f"https://rest.uniprot.org/uniprotkb/{accession_number}.fasta"

  response = requests.get(url)
  response.raise_for_status()
  fasta_text = response.text
  lines = fasta_text.strip().split("\n")

  header = lines[0]           # the ">sp|P68431|H31_HUMAN ..." line
  sequence = "".join(lines[1:])  # join all sequence lines into one string

  return header, sequence

accession_number = "P68431"
header, h31_ref = fetch_uniprot_sequence(accession_number)
print(header)
print(h31_ref)

>sp|P68431|H31_HUMAN Histone H3.1 OS=Homo sapiens OX=9606 GN=H3C1 PE=1 SV=2
MARTKQTARKSTGGKAPRKQLATKAARKSAPATGGVKKPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQDFKTDLRFQSSAVMALQEACEAYLVGLFEDTNLCAIHAKRVTIMPKDIQLARRIRGERA


With the protein sequence obtained, the polypeptide chains are now ready to match with the reference.

In [58]:
for model in structure:
    for chain in model:
        peptides = ppb.build_peptides(chain)
        for pep in peptides:
            chain_seq = str(pep.get_sequence())

            if chain_seq in h31_ref:
                match = "EXACT SUBSTRING MATCH — likely H3.1"
            else:
                match = "no match"

            print(f"Chain {chain.id}: {len(chain_seq)} aa | {match}")
            print(f"  {chain_seq[:40]}...")

Chain A: 97 aa | EXACT SUBSTRING MATCH — likely H3.1
  PHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQD...
Chain B: 78 aa | no match
  NIQGITKPAIRRLARRGGVKRISGLIYEETRGVLKVFLEN...
Chain C: 108 aa | no match
  RAKAKTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVY...
Chain D: 96 aa | no match
  KRSRKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDI...
Chain E: 99 aa | EXACT SUBSTRING MATCH — likely H3.1
  KPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQ...
Chain F: 84 aa | no match
  RKVLRDNIQGITKPAIRRLARRGGVKRISGLIYEETRGVL...
Chain G: 104 aa | no match
  KTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVYLAAV...
Chain H: 92 aa | no match
  RKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDIFER...


In [59]:
def compare_chains_with_ref(ref_seq):

  for model in structure:
      for chain in model:
          peptides = ppb.build_peptides(chain)
          for pep in peptides:
              chain_seq = str(pep.get_sequence())

              if chain_seq in (ref_seq):
                  match = "EXACT SUBSTRING MATCH — likely H3.1"
              else:
                  match = "no match"

              print(f"Chain {chain.id}: {len(chain_seq)} aa | {match}")
              print(f"  {chain_seq[:40]}...")

In [60]:
compare_chains_with_ref(h31_ref)

Chain A: 97 aa | EXACT SUBSTRING MATCH — likely H3.1
  PHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQD...
Chain B: 78 aa | no match
  NIQGITKPAIRRLARRGGVKRISGLIYEETRGVLKVFLEN...
Chain C: 108 aa | no match
  RAKAKTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVY...
Chain D: 96 aa | no match
  KRSRKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDI...
Chain E: 99 aa | EXACT SUBSTRING MATCH — likely H3.1
  KPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQ...
Chain F: 84 aa | no match
  RKVLRDNIQGITKPAIRRLARRGGVKRISGLIYEETRGVL...
Chain G: 104 aa | no match
  KTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVYLAAV...
Chain H: 92 aa | no match
  RKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDIFER...


In [61]:
from Bio.PDB import PDBIO, Select

class ChainSelect(Select):      # to create a sieve
    def accept_chain(self, chain):
      '''Only the ones that return true will stay in the sieve when the rest are filtered out.'''
      return chain.id in ["A", "E"]

io = PDBIO()
io.set_structure(structure)
io.save("3AFA_H3.1.pdb", ChainSelect())

Verifying the file 3AFA_H3.1.pdb is actually created.

In [62]:
p = Path("3AFA_H3.1.pdb")
print(p.exists(), p.stat().st_size, "bytes")

True 135522 bytes


Visualising H3.1 structure.

In [63]:
with open("3AFA_H3.1.pdb") as f:
  pdb_data = f. read()

  # () calls the "read" method

view = py3Dmol.view(width = 500, height = 400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Repeat this process to download a new pdb file, isolate the histone H3.3 and visualise its strucure**

In [64]:
download_pdb("3AV2")

Downloaded 3AV2.pdb


In [65]:
p = Path("3AV2.pdb")
# The quote refers to a string. Without the quote, the method would look for a variable called 3AV2.pdb, which does not exist.

print(p.exists(), p.stat().st_size, "bytes")
# The full chain p.stat().st_size means: "call stat() on p to get a stat-result object,
# then read the st_size attribute off that object" — which gives you the file size in bytes.

True 1042551 bytes


In [66]:
with open("3AV2.pdb") as f:
  pdb_data = f. read()

view = py3Dmol.view(width = 500, height = 400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [73]:
structure = load_structure("3AV2","3AV2.pdb")
get_sequence(structure)

Chain A: 103 residues, sequence: PHRYRPGTVALREIRRYQKS...
Chain B: 83 residues, sequence: NIQGITKPAIRRLARRGGVK...
Chain C: 115 residues, sequence: AKTRSSRAGLQFPVGRVHRL...
Chain D: 98 residues, sequence: RSRKESYSIYVYKVLKQVHP...
Chain E: 116 residues, sequence: PHRYRPGTVALREIRRYQKS...
Chain F: 98 residues, sequence: RKVLRDNIQGITKPAIRRLA...
Chain G: 109 residues, sequence: KTRSSRAGLQFPVGRVHRLL...
Chain H: 93 residues, sequence: RKESYSIYVYKVLKQVHPDT...


In [74]:
search_uniprot("Histone H3.3")

P84243: Histone H3.3
O43463: Histone-lysine N-methyltransferase SUV39H1
Q9Y294: Histone chaperone ASF1A
Q9NVP2: Histone chaperone ASF1B
Q9H5I1: Histone-lysine N-methyltransferase SUV39H2


In [75]:
fetch_uniprot_sequence("P84243")

('>sp|P84243|H33_HUMAN Histone H3.3 OS=Homo sapiens OX=9606 GN=H3-3A PE=1 SV=2',
 'MARTKQTARKSTGGKAPRKQLATKAARKSAPSTGGVKKPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQDFKTDLRFQSAAIGALQEASEAYLVGLFEDTNLCAIHAKRVTIMPKDIQLARRIRGERA')

In [76]:
h33_ref = fetch_uniprot_sequence("P84243")
compare_chains_with_ref(h33_ref)

Chain A: 97 aa | no match
  PHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQD...
Chain B: 78 aa | no match
  NIQGITKPAIRRLARRGGVKRISGLIYEETRGVLKVFLEN...
Chain C: 105 aa | no match
  AKTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVYLAA...
Chain D: 94 aa | no match
  RSRKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDIF...
Chain E: 98 aa | no match
  PHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQD...
Chain F: 84 aa | no match
  RKVLRDNIQGITKPAIRRLARRGGVKRISGLIYEETRGVL...
Chain G: 104 aa | no match
  KTRSSRAGLQFPVGRVHRLLRKGNYSERVGAGAPVYLAAV...
Chain H: 92 aa | no match
  RKESYSIYVYKVLKQVHPDTGISSKAMGIMNSFVNDIFER...
